# 4. TUR — tek geçişte eğitim

**Baştan sona sırayla çalıştır.** Zıplama yok, atlanacak hücre yok.

`Runtime → Run all` de çalışır, ama **9. adımda durup çıktıyı kontrol et**
(araç dağılımı) — orada bir sorun varsa eğitime girmenin anlamı yok.

Bu defter `train_lora.ipynb`'den yalnızca 4. tur için gereken hücreleri
alır. Kod birebir aynı; değişen tek şey sıra ve gereksizlerin çıkarılmış
olması.

| adım | ne | süre |
|---|---|---|
| 1-4 | ortam + veri kontrolü | ~1 dk |
| 5 | unsloth kurulumu | ~3 dk |
| 6 | taban model + LoRA | ~3 dk |
| 7-10 | veri hazırlığı | ~1 dk |
| 11-13 | eğitici + devam testi | ~5 dk |
| 14 | **eğitim** | ~45 dk |
| 15-16 | kayıp eğrisi + kaydet | ~1 dk |

⚠️ 14. adımda sekmeyi kapatma, bilgisayarı uyutma.

## 1. GPU

In [ ]:
import torch

assert torch.cuda.is_available(), 'GPU yok! Runtime > Change runtime type > T4 GPU'

ad = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'GPU  : {ad}')
print(f'VRAM : {vram:.1f} GB')
if 'T4' not in ad:
    print(f'{chr(10)}NOT: T4 bekleniyordu, {ad} bulundu. Sureler farkli cikar.')


## 2. Drive'ı bağla

Açılan pencereden hesabını seç, izin ver.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')


## 3. Klasörler

In [ ]:
from pathlib import Path

KOK = Path('/content/drive/MyDrive/codifya')
VERI = KOK / 'veri'
CHECKPOINT = KOK / 'checkpoint'
CIKTI = KOK / 'cikti'

for d in (VERI, CHECKPOINT, CIKTI):
    d.mkdir(parents=True, exist_ok=True)
    print(f'{d!s:45s} hazir')


## 4. Veri yerinde mi

`router_train.jsonl` **3715**, `gerekce_train.jsonl` **40293** satır olmalı.

In [ ]:
BEKLENEN = {
    'gerekce_train.jsonl': 40293,
    'gerekce_val.jsonl': None,
    'gerekce_test.jsonl': None,
    # ⚠️ 43798'di — sinif dengeleme + tekillestirme hatti degistikten sonra
    # gecersiz kaldi ve her kosuda yanlis alarm veriyordu. 4. tur verisinde
    # 3715. Bu sayi veri her yeniden uretildiginde degisir; yanlis cikarsa
    # once `veri_bolme` ciktisiyla karsilastir, panige kapilma.
    'router_train.jsonl': 3715,
    'router_val.jsonl': None,
    'router_test.jsonl': None,
    'golden_set_aday.jsonl': None,
}

eksik = []
for ad_, beklenen in BEKLENEN.items():
    yol = VERI / ad_
    if not yol.exists():
        eksik.append(ad_)
        print(f'  YOK  {ad_}')
        continue
    mb = yol.stat().st_size / 1024**2
    with yol.open(encoding='utf-8') as f:
        n = sum(1 for _ in f)
    uyari = ''
    if beklenen and n != beklenen:
        uyari = f'  <-- {beklenen} bekleniyordu'
    print(f'  VAR  {ad_:24s} {mb:7.1f} MB  {n:6d} satir{uyari}')

if eksik:
    raise SystemExit(f'Eksik dosya: {eksik} -- Drive/codifya/veri/ klasorune yukle.')
print('Veri tamam.')


## 5. unsloth kurulumu

~3 dakika, sessiz çalışır.

In [ ]:
!pip install -q unsloth


## 6. Taban model + LoRA

Sıfırdan yükleniyor. `trainable%` %1'in altında olmalı.

In [ ]:
from unsloth import FastLanguageModel

MODEL = 'unsloth/Qwen2.5-1.5B-Instruct'
MAX_SEQ = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL,
    max_seq_length=MAX_SEQ,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
)
model.print_trainable_parameters()


## 7. Yardımcı fonksiyonlar

İstem biçimini kuran fonksiyonlar. `GOREV:` etiketi modele hangi işi
yaptığını söylüyor — router mı, gerekçe mi.

In [ ]:
import json
import random

from datasets import Dataset

random.seed(42)


def satirlari_oku(ad, sinir=None):
    with (VERI / ad).open(encoding='utf-8') as f:
        return [json.loads(s) for s in (f if sinir is None else [x for _, x in zip(range(sinir), f, strict=False)])]


def gerekce_metni(k):
    return 'GOREV: gerekce' + chr(10) + k['istem'] + chr(10) + k['cevap']


def router_metni(k):
    cevap = json.dumps(
        {'arac': k['arac'], 'parametreler': k.get('parametreler', {})},
        ensure_ascii=False,
    )
    return 'GOREV: router' + chr(10) + 'SORU: ' + k['soru'] + chr(10) * 2 + 'ARAC:' + chr(10) + cevap


print('yardimci fonksiyonlar hazir')


## 8. Doğrulama kümesi

500 gerekçe + 500 router. Ezberleme buradan anlaşılacak.

In [ ]:
gv = [gerekce_metni(k) for k in satirlari_oku('gerekce_val.jsonl', 500)]
rv = [router_metni(k) for k in satirlari_oku('router_val.jsonl', 500)]

dogrulama = Dataset.from_dict({'text': [m + tokenizer.eos_token for m in gv + rv]})
print('dogrulama ornegi:', len(dogrulama))


## 9. ⭐ Dengeli örnekleme — BURADA DUR

1. turda router dosyasının ilk 7.500 satırı alınmıştı ve dosya araca göre
sıralı olduğu için **üç araç eğitime hiç girmedi**; router doğruluğu
%70'ten %40'a düştü.

Aşağıdaki iki hücrenin çıktısında görmen gereken: **`disarida kalan arac: 0`**

In [ ]:
import json
import random
from collections import Counter, defaultdict

# Yeni (dengelenmis) veriyle olculdu -- ayrinti yukaridaki markdown'da.
TAVAN_KAT = 20
ROUTER_ORNEK = 2_000
GEREKCE_ORNEK = 8_000


def dengeli_kota(mevcut, toplam, tavan_kat=TAVAN_KAT):
    """Esit paya yakin dagitir, her sinifi kapasitesiyle sinirlar."""
    kalan = dict(mevcut)
    kota = {}
    butce = toplam
    while kalan and butce > 0:
        pay = butce // len(kalan)
        if pay == 0:
            break
        doyan = {a: m * tavan_kat for a, m in kalan.items() if m * tavan_kat <= pay}
        if not doyan:
            for a in kalan:
                kota[a] = pay
            break
        for a, k in doyan.items():
            kota[a] = k
            butce -= k
            del kalan[a]
    return kota


def dengeli_ornekle(kayitlar, anahtar, toplam, tohum=42):
    rnd = random.Random(tohum)
    gruplar = defaultdict(list)
    for k in kayitlar:
        gruplar[k[anahtar]].append(k)
    kota = dengeli_kota({a: len(v) for a, v in gruplar.items()}, toplam)
    secilen = []
    for a, adet in kota.items():
        havuz = gruplar[a]
        if adet <= len(havuz):
            secilen += rnd.sample(havuz, adet)
        else:
            secilen += havuz * (adet // len(havuz)) + rnd.sample(havuz, adet % len(havuz))
    rnd.shuffle(secilen)
    return secilen


# ⚠️ TUM dosya okunuyor -- 1. turdaki hata buradaydi (ilk N satir aliniyordu)
with (VERI / 'router_train.jsonl').open(encoding='utf-8') as f:
    router_hepsi = [json.loads(s) for s in f]
with (VERI / 'gerekce_train.jsonl').open(encoding='utf-8') as f:
    gerekce_hepsi = [json.loads(s) for s in f]

print('okunan  router :', len(router_hepsi))
print('okunan  gerekce:', len(gerekce_hepsi))


In [ ]:
router_sec = dengeli_ornekle(router_hepsi, 'arac', ROUTER_ORNEK)
gerekce_sec = dengeli_ornekle(gerekce_hepsi, 'karar_tipi', GEREKCE_ORNEK)

print('ROUTER  arac dagilimi')
print('-' * 58)
mevcut = Counter(k['arac'] for k in router_hepsi)
yeni = Counter(k['arac'] for k in router_sec)
for a in sorted(mevcut):
    print(f'  {a:32s} {mevcut[a]:6d} -> {yeni[a]:5d}  ({yeni[a] / mevcut[a]:.1f}x)')
print(f'  disarida kalan arac: {sum(1 for a in mevcut if yeni[a] == 0)}')
print(f'  ozgun soru: {len({k[chr(115) + chr(111) + chr(114) + chr(117)] for k in router_sec})}/{len(router_sec)}')
print()
print('GEREKCE  karar tipi dagilimi')
print('-' * 58)
gm = Counter(k['karar_tipi'] for k in gerekce_hepsi)
gy = Counter(k['karar_tipi'] for k in gerekce_sec)
for t in sorted(gm):
    print(f'  {t:32s} {gm[t]:6d} -> {gy[t]:5d}')


In [ ]:
from datasets import Dataset

metinler = [gerekce_metni(k) for k in gerekce_sec] + [router_metni(k) for k in router_sec]
random.Random(42).shuffle(metinler)
metinler = [m + tokenizer.eos_token for m in metinler]

egitim = Dataset.from_dict({'text': metinler})
print(len(egitim), 'ornek hazir (2. tur)')
print('-' * 70)
print(egitim[0]['text'][:350])


## 10. Tur etiketi

Checkpoint dizini bundan türer. Her turda değişmeli.

In [ ]:
# ⚠️ 4. TUR (2026-08-08). Iki veri duzeltmesi:
#  1) gecelik_ozet_sorgula parametresiz kapsamasi 184/0 -> 184/35
#     (3. turda bu araçta tam dogruluk 7/10 -> 1/10 dusmustu)
#  2) parametresiz sablon 15 -> 44, olcum setine yakinlik kapisi eklendi
# Ornek sayisi ve ayarlar 3. turla AYNI -- tek degisken veri olsun diye.
#
# ⚠️⚠️ ETIKETI HER TURDA DEGISTIR. Ayni kalirsa checkpoint dizini de ayni
# kalir ve egitim hucresi taban modelden degil ONCEKI TURDAN devam eder.
#
# ⚠️ Bu hucre YALNIZCA etiketi koyar. Onceden `CIKIS` ve `ARGS['output_dir']`
# da burada yaziliyordu ve dongusel bir bagimlilik kuruyordu: `ARGS` egitici
# hucresinde tanimli, o hucre de `TUR_ETIKETI`'ni bekliyor. Yani hangisini
# once kosarsan kos biri eksik kaliyordu. Artik tek yon var:
#     bu hucre  ->  egitici hucresi (CIKIS'i etiketten turetir)
TUR_ETIKETI = 'tur4'
print('tur etiketi:', TUR_ETIKETI)
print('egitici hucresi checkpoint dizinini bundan turetecek: b33_' + TUR_ETIKETI)


## 11. Eğitici

⚠️ Çıktıda **`b33_tur4`** yazmalı. Başka bir şey yazıyorsa dur.

In [ ]:
from trl import SFTTrainer

# ⚠️ ETIKET BURADA SABIT YAZILMAZ. Onceden 'b33_tur1' gomuluydu ve su
# tuzagi kuruyordu: tur yapilandirma hucresi (asagida, 2. tur bolumunde)
# TUR_ETIKETI'ni degistirip ARGS['output_dir']'i guncelliyor, ama SONRA bu
# hucre yeniden kosuldugunda CIKIS tekrar 'b33_tur1'e donuyordu. O dizinde
# 1. turun checkpoint'leri duruyor -> egitim hucresindeki `devam` kontrolu
# True cikip **1. TURDAN DEVAM EDIYORDU**. Sonuc: taban modelden degil eski
# bir turdan devam eden, olculemez bir model + ustune yazilmis checkpointler.
TUR_ETIKETI = globals().get('TUR_ETIKETI', 'tur1')
CIKIS = str(CHECKPOINT / f'b33_{TUR_ETIKETI}')

ARGS = dict(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=20,
    num_train_epochs=1,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=25,
    optim='adamw_8bit',
    weight_decay=0.01,
    lr_scheduler_type='linear',
    seed=42,
    output_dir=CIKIS,
    report_to='none',
    save_steps=200,
    save_total_limit=2,
    eval_strategy='steps',
    eval_steps=200,
)


def egitici_kur():
    try:
        from trl import SFTConfig

        return SFTTrainer(
            model=model,
            train_dataset=egitim,
            eval_dataset=dogrulama,
            args=SFTConfig(dataset_text_field='text', max_seq_length=MAX_SEQ, **ARGS),
        )
    except (ImportError, TypeError) as e:
        print('SFTConfig kullanilamadi:', type(e).__name__)
        from transformers import TrainingArguments

        return SFTTrainer(
            model=model,
            tokenizer=tokenizer,
            train_dataset=egitim,
            eval_dataset=dogrulama,
            dataset_text_field='text',
            max_seq_length=MAX_SEQ,
            args=TrainingArguments(**ARGS),
        )


trainer = egitici_kur()
adim = len(egitim) // (2 * 4)
print('toplam adim (1 epoch):', adim)
print('tahmini sure          :', round(adim * 2.1 / 60), 'dakika')


## 12. Devam-etme testi

40 adım koş → dur → checkpoint'ten devam et → sayacın 40'tan devam
ettiğini gör. Oturum koparsa saatler kazandırır.

⚠️ Bu test modeli 80 adım eğitiyor, yani asıl eğitim tam sıfırdan
başlamıyor. Üç turdur böyle — karşılaştırma adil.

In [ ]:
import shutil

TEST_CIKIS = str(CHECKPOINT / 'b33_devam_testi')
shutil.rmtree(TEST_CIKIS, ignore_errors=True)

TEST_ARGS = {**ARGS, 'output_dir': TEST_CIKIS, 'max_steps': 40, 'save_steps': 20,
             'eval_strategy': 'no', 'num_train_epochs': 1, 'logging_steps': 20}


def test_egitici(args):
    try:
        from trl import SFTConfig

        return SFTTrainer(model=model, train_dataset=egitim,
                          args=SFTConfig(dataset_text_field='text', max_seq_length=MAX_SEQ, **args))
    except (ImportError, TypeError):
        from transformers import TrainingArguments

        return SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=egitim,
                          dataset_text_field='text', max_seq_length=MAX_SEQ,
                          args=TrainingArguments(**args))


t1 = test_egitici(TEST_ARGS)
s1 = t1.train()
print('1. kosu bitti, adim:', s1.global_step)


In [ ]:
import os

kayitlar = sorted(d for d in os.listdir(TEST_CIKIS) if d.startswith('checkpoint-'))
print('Drive\'daki checkpointler:', kayitlar)

t2 = test_egitici({**TEST_ARGS, 'max_steps': 80})
s2 = t2.train(resume_from_checkpoint=True)

print()
print('=' * 60)
print('1. kosu bitis adimi :', s1.global_step)
print('2. kosu bitis adimi :', s2.global_step)
gecti = s2.global_step > s1.global_step and len(kayitlar) > 0
print('DEVAM ETME:', 'CALISIYOR' if gecti else 'CALISMIYOR -- uzun egitime girme!')
print('=' * 60)


## 13. ⭐ EĞİTİM (~45 dk)

⚠️ Çıktının ilk satırında **`onceki checkpoint var mi: False`** yazmalı.
`True` yazıyorsa **DUR** — eski bir turdan devam etmeye çalışıyor.

Sekmeyi kapatma, bilgisayarı uyutma. Kopma olursa checkpoint'ten devam
eder, en fazla 7 dakika kaybedilir.

In [ ]:
import os

devam = os.path.isdir(CIKIS) and any(d.startswith('checkpoint-') for d in os.listdir(CIKIS))
print('onceki checkpoint var mi:', devam)

sonuc = trainer.train(resume_from_checkpoint=devam)
print(sonuc)


## 14. Kayıp eğrisi

Doğrulama kaybı düşüp yatay seyre geçmeli. Yükseliyorsa ezberliyor.

In [ ]:
kayit = trainer.state.log_history
egitim_kaybi = [(k['step'], k['loss']) for k in kayit if 'loss' in k]
dogrulama_kaybi = [(k['step'], k['eval_loss']) for k in kayit if 'eval_loss' in k]

print('adim   egitim   dogrulama')
print('-' * 32)
d = dict(dogrulama_kaybi)
for adim_, kayip in egitim_kaybi:
    dk = d.get(adim_)
    print(f'{adim_:5d}  {kayip:7.4f}  {dk if dk is None else round(dk, 4)}')

if len(dogrulama_kaybi) >= 2:
    ilk, son = dogrulama_kaybi[0][1], dogrulama_kaybi[-1][1]
    en_iyi = min(k for _, k in dogrulama_kaybi)
    print()
    print('ilk dogrulama kaybi :', round(ilk, 4))
    print('son dogrulama kaybi :', round(son, 4))
    print('en iyi              :', round(en_iyi, 4))
    print('EZBERLEME UYARISI' if son > en_iyi * 1.05 else 'Ezberleme isareti yok')


## 15. LoRA'yı Drive'a kaydet

`cikti/b33_tur4_lora/` altına ~50 MB.

In [ ]:
# ⚠️ Tur etiketi asagidaki tur yapilandirma hucresinde belirlenir.
# Tanimli degilse 1. tur kosuluyor demektir.
TUR_ETIKETI = globals().get('TUR_ETIKETI', 'tur1')
TUR1 = CIKTI / f'b33_{TUR_ETIKETI}_lora'

model.save_pretrained(str(TUR1))
tokenizer.save_pretrained(str(TUR1))

import os

boyut = sum(os.path.getsize(os.path.join(k, d)) for k, _, dl in os.walk(TUR1) for d in dl)
print('kaydedildi:', TUR1)
print('boyut     :', round(boyut / 1024**2, 1), 'MB')


## Bitti

Sonraki adım **`b34_gguf_colab.ipynb`** — LoRA'yı taban modelle birleştirip
GGUF'a çevirir (~15 dk). Orada `TUR_ETIKETI`'ni yine `tur4` seçmen gerekecek.

Ölçüm Colab'da **yapılmayacak** — yerelde Ollama üzerinden yapılıyor,
taban çizgiyle karşılaştırılabilir olması için.